In [29]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [3]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str




In [4]:
workflow = StateGraph(GraphState) 

In [5]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [6]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [7]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [8]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.

    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [9]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [10]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [11]:
def transcribe_audio(audio_path):
    audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

    return transcription.text
    

user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
print(user_query)

Why does it help to keep my butt on the bench once the bar is out?


In [12]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [13]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_07.mov"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/bench_07
Video name: bench_07
Bench Press

1. Feet & base
2. Leg drive
3. Glutes & hips
4. Lower back & arch
5. Shoulder blades & back alignment
6. Grip & wrist position
7. Bar path
8. Elbow position
9. Head & neck
10. Lockout position
11. Tempo and control
Great arch and you’re controlling the bar through full reps. A few fixes will make this stronger and safer.

What I’m seeing
- Heels are up and your feet are set far forward. Your shins aren’t near‑vertical, so you can’t get efficient leg drive.
- Your legs are flared out from the bench and your lower body looks loose between reps.
- Hips look like they’re trying to rise as you press.
- Elbows are tending to flare; bar path goes mostly straight up rather than slightly back toward the rack.

How to fix it (use these cues)
- Foot position and leg drive
  - Plant your heels flat against the floor. Bring each foot back as far as is comfortable so your shins are nearly vertical, a

In [14]:

result = app.invoke({
    "session_id": 123,
    "user_query": user_query,
    "user_video": ""
})

print(result["response"])

Keeping your butt down lets your leg drive do its job.

- With your heels planted, the force from the legs travels along the bench through the hips into the arched back. That reinforces the arch and keeps your chest in its high position so you stay tight and press more efficiently.
- If your butt lifts, you break that chain. Your core goes soft, you lose your footing, and you’re more likely to slide forward on a slippery bench—dangerous when the weight gets heavy.
- Foot cue: set your feet so your shins are nearly vertical and keep the heels flat. This lets you use leg drive without “bridging” your hips off the pad.

If your bench is slick, wrap a couple of bungees around the seat so your butt sticks and stays down.


In [15]:
result = app.invoke({
    "session_id": 123,
    "user_query": "thanks for your help today!",
    "user_video": ""
})

print(result["response"])

You're welcome! If you have any more questions or need further assistance, feel free to reach out. Happy lifting! 🏋️‍♂️


In [16]:
result = app.invoke({
    "session_id": 123,
    "user_query": "can you tell me again how to make it easier?",
    "user_video": ""
})

print(result["response"])

Certainly! Here’s a recap to make your bench press easier and more effective:

### Foot Position and Leg Drive
- **Heels Down**: Keep your heels firmly planted on the floor.
- **Foot Placement**: Bring your feet back so your shins are nearly vertical. This helps with stability and leg drive.
- **Leg Drive**: Push through your heels during the press to help maintain your arch and stability.

### Setup and Tightness
- **Four Points of Contact**: Keep your head, upper back, glutes, and feet firmly planted.
- **Shoulder Blades**: Pinch them together to create a stable base.
- **Grip**: Hold the bar tightly, imagining you’re trying to bend it, which engages your lats and stabilizes your shoulders.

### Unrack and Execution
- **Unrack Properly**: Lift the bar out, not up, to maintain your setup.
- **Breathing and Bracing**: Take a deep breath into your belly and brace your core before each rep.
- **Elbow Position**: Keep your elbows at about a 45-degree angle to your body.
- **Bar Path**: Lo

In [17]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [28]:
def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": "eval-session",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    state = {**state, **vector_db_node(state)}
    state = {**state, **response_generator(state)}
    

In [30]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

results = evaluate(
    run_rag_pipeline,
    data="Image assessment response accuracy v1",
    evaluators=[correctness_evaluator],
    experiment_prefix="vision-faithfullness-rag-v1"
)

View the evaluation results for experiment: 'vision-faithfullness-rag-v1-2686bbe9' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=7d3d81e8-ad2d-4b03-8879-6db85f6833e3




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to processed-images/tmpcsjct0rf
Video name: tmpcsjct0rf
**Bench Press**

1. Feet & base
2. Glutes & back
3. Core & ribcage
4. Shoulder position
5. Elbow alignment
6. Wrist position
7. Bar path
8. Head & neck


Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 019cd839-9a5a-7832-9561-6981a6056fc3: KeyError('answer')
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1601, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/evaluator.py", line 332, in evaluate_run
    result = self.func(
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_35863/889772864.py", line 2, in correctness_evaluator
    generated = run.outputs["answer"]
KeyError: 'answer'


Frames processed: 15, (10s cap)
Saved to processed-images/tmp4cye7hjr
Video name: tmp4cye7hjr
**Dumbbell Bench Press**

1. Feet & base
2. Hips & glutes
3. Lower back position
4. Shoulder position
5. Elbow alignment
6. Wrist position
7. Dumbbell path
8. Head & neck stability
9. Tempo and control


...

Frames processed: 15, (10s cap)
Saved to processed-images/tmpe0612y9c
Video name: tmpe0612y9c
Frames processed: 15, (10s cap)
Saved to processed-images/tmp45m7d1a4
Video name: tmp45m7d1a4
**Bench Press**

1. Feet & base
2. Lower back & arch
3. Glutes & hips
4. Core engagement
5. Shoulder position
6. Grip & wrist alignment
7. Bar path
8. Head & Neck
9. Tempo and control


Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 019cd841-7fe2-7e60-a90c-19336eef10bd: KeyError('answer')
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1601, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/evaluator.py", line 332, in evaluate_run
    result = self.func(
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_35863/889772864.py", line 2, in correctness_evaluator
    generated = run.outputs["answer"]
KeyError: 'answer'


Frames processed: 15, (10s cap)
Saved to processed-images/tmprj6xvclr
Video name: tmprj6xvclr
**Bench Press**

1. Feet & base
2. Leg drive
3. Glutes & hips
4. Lower back arch
5. Grip width
6. Wrist position
7. Elbow angle
8. Bar path
9. Head & neck position
10. Lockout position


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19959663
API Key: lsv2_********************************************cctrace=019cd842-eeec-7dc1-98aa-e0f39153644a,id=019cd842-fd2b-7ea0-b3f1-30acd68484ce
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19958637
API Key: ls

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19955083
API Key: lsv2_********************************************cctrace=019cd842-eeec-7dc1-98aa-e0f39153644a,id=019cd842-fd7a-75b0-af97-8cb74b93105c; trace=019cd842-eeec-7dc1-98aa-e0f39153644a,id=019cd844-34ed-7c23-bcd7-20947f36f9ba
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLErr